#IMPOPRTING LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf

from datetime import date
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#LOADING DATA

In [ ]:
START = "2015-01-01"
TODAY = date.today().strftime("%Y-%m-%d")

def load_data(ticker):
    df = yf.download(ticker, START, TODAY)
    df = df[['Close']]
    df.dropna(inplace=True)
    return df

df = load_data("AAPL")
df.head()

#TRAIN-TEST SPLIT

In [ ]:
train_size = int(len(df) * 0.7)

train = df[:train_size]
test = df[train_size:]

print(train.shape, test.shape)
df.head()

#FEATURE SCALING

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

#SEQUENCE WINDOW CREATION

In [ ]:
window = 100

def create_sequences(data):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled)

# For test we must include last 100 days of train
combined = np.vstack((train_scaled[-window:], test_scaled))
X_test, y_test = create_sequences(combined)

print(X_train.shape, X_test.shape)

#BUILDING LSTM MODEL AND COMPILING

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(window,1)),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

#TRAINING

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

#Loss Graphs

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()
plt.title("Training vs Validation Loss")
plt.savefig("loss_curve.png", dpi=300, bbox_inches="tight")
plt.show()

# Model Prediction and Inverse Scaling

In [ ]:
y_pred = model.predict(X_test)

# Inverse transform
y_pred = scaler.inverse_transform(y_pred)
y_test_actual = scaler.inverse_transform(y_test)

# Model Evaluation Metrics (Mean Absolute Error, Mean Squared Error, and R-squared)

In [ ]:
mae = mean_absolute_error(y_test_actual, y_pred)
mse = mean_squared_error(y_test_actual, y_pred)
r2 = r2_score(y_test_actual, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2:", r2)

#Prediction Vs Actual

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(y_test_actual, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("Actual vs Predicted Price")
plt.savefig("prediction_plot.png", dpi=300, bbox_inches="tight")
plt.show()

#GRU Model and compiling

In [ ]:
gru_model = tf.keras.Sequential([
    tf.keras.layers.GRU(64, return_sequences=True, input_shape=(window,1)),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.GRU(64, return_sequences=True),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.GRU(32),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(1)
])

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

gru_model.summary()

# TRAINING

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

gru_history = gru_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

#Model Prediction and Inverse Scaling

In [ ]:
gru_pred = gru_model.predict(X_test)

gru_pred = scaler.inverse_transform(gru_pred)
y_test_actual = scaler.inverse_transform(y_test)

# Model Architecture Comparison: LSTM vs GRU

In [ ]:
# LSTM metrics
lstm_mae = mean_absolute_error(y_test_actual, y_pred)
lstm_r2 = r2_score(y_test_actual, y_pred)

# GRU metrics
gru_mae = mean_absolute_error(y_test_actual, gru_pred)
gru_r2 = r2_score(y_test_actual, gru_pred)

print("LSTM -> MAE:", lstm_mae, " R2:", lstm_r2)
print("GRU  -> MAE:", gru_mae, " R2:", gru_r2)

#LSTM Vs GRU

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(y_test_actual, label="Actual")
plt.plot(y_pred, label="LSTM")
plt.plot(gru_pred, label="GRU")
plt.legend()
plt.title("LSTM vs GRU Comparison")
plt.savefig("lstm_vs_gru.png", dpi=300, bbox_inches="tight")
plt.show()

# Ablation Study: LSTM Training with Different Window Sizes

In [ ]:
def train_lstm_with_window(window_size):

    # Create sequences
    def create_sequences(data):
        X, y = [], []
        for i in range(window_size, len(data)):
            X.append(data[i-window_size:i])
            y.append(data[i])
        return np.array(X), np.array(y)

    X_train, y_train = create_sequences(train_scaled)

    combined = np.vstack((train_scaled[-window_size:], test_scaled))
    X_test, y_test = create_sequences(combined)

    # Build model
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(window_size,1)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse')

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    preds = model.predict(X_test)
    preds = scaler.inverse_transform(preds)
    y_actual = scaler.inverse_transform(y_test)

    mae = mean_absolute_error(y_actual, preds)
    r2 = r2_score(y_actual, preds)

    return mae, r2

#Mean Absolute Error for different Window Sizes

In [ ]:
mae_100, r2_100 = train_lstm_with_window(100)
mae_30, r2_30 = train_lstm_with_window(30)

print("Window 100 -> MAE:", mae_100, " R2:", r2_100)
print("Window 30  -> MAE:", mae_30, " R2:", r2_30)

#PREDICTION ERROR

In [ ]:
errors = y_test_actual.flatten() - y_pred.flatten()

plt.figure(figsize=(10,5))
plt.plot(errors)
plt.title("Prediction Error Over Time")
plt.savefig("prediction_error.png", dpi=300, bbox_inches="tight")
plt.show()

print("Max Error:", np.max(np.abs(errors)))

#DIRECTIONAL ACCURACY

In [ ]:
actual_direction = np.sign(np.diff(y_test_actual.flatten()))
pred_direction = np.sign(np.diff(y_pred.flatten()))

directional_accuracy = np.mean(actual_direction == pred_direction) * 100

print("Directional Accuracy (%):", directional_accuracy)